# ANALYSIS OF WADDELL SPUTNUM DATA

In [1]:
import numpy as np
import pandas as pd
import os

In [9]:
PROJECT_DIR = os.path.dirname(os.path.abspath("."))
DATA_DIR = os.path.join(PROJECT_DIR, "data")
DATA_DIR = os.path.join(DATA_DIR, "samples")
PATH = os.path.join(DATA_DIR, 'Sputa_transcriptomes_Waddell.xlsx')
DF = pd.read_excel(PATH)
del DF["Unnamed: 0"]
DF.head()

,TargetID,GeneName,Description,Gene_ID,4557_d0,4557_d3,4557_wk1,4557_wk2,4558_d0,4558_d3,...,4877_wk2,4936_d0,4936_wk1,4936_wk2,4939_d0,4939_wk1,4939_wk2,5755_d0,5755_d3,5755_wk1
0,BCG_0014c,"TBFG_10014, TBFG_00014",transmembrane serine/threonine-protein kinase ...,BUGS0000000580609|BCG_0014c,7.521097,7.413344,7.334556,7.407448,7.393876,7.441137,...,7.525532,7.575306,7.434187,7.549169,7.563523,7.498329,7.529780,7.552024,7.524677,7.574339
1,BCG_0067c,"TBFG_10035, TBFG_00035",hypothetical protein,BUGS0000000580289|BCG_0067c,10.113032,10.432085,10.876276,9.853786,10.226141,9.882812,...,10.600699,10.600699,10.681497,10.586504,10.839663,10.379874,10.778040,10.312051,10.536067,10.003142
2,BCG_1596,NaN,BCG_1596 protein,BUGS0000000579731|BCG_1596,8.103065,8.223157,8.160017,8.194219,8.291422,8.182801,...,8.435111,8.221434,8.397545,8.621949,8.463405,8.623645,8.594069,8.431190,8.294412,8.337461
3,BCG_3354c,lat',Probable L-lysine-epsilon aminotransferase lat',BUGS0000000580485|BCG_3354c,8.382316,8.449695,8.129300,8.168247,8.049271,8.309573,...,8.098397,8.239758,8.779252,7.988193,8.233984,8.192326,8.415045,8.665665,8.525736,8.402685
4,JTY_0020,"TBFG_10020, TBFG_00020",hypothetical protein,BUGS0000000573369|JTY_0020,7.940714,7.850831,7.652419,7.760914,7.714380,7.839520,...,8.268402,8.336970,8.063147,8.212019,8.163627,8.289392,7.773844,7.979736,7.841713,8.038801


In [17]:
# Create separate DFs for patient data and gene descriptions
DESCRIPTION_DF = DF[ ["TargetID", "GeneName", "Description", "Gene_ID"]]
columns = list(DF.columns)
columns.remove("GeneName")
columns.remove("Description")
columns.remove("Gene_ID")
COUNT_DF = DF[columns]
COUNT_DF = COUNT_DF.set_index("TargetID")
columns = list(COUNT_DF.columns)
COUNT_DF.head()

,4557_d0,4557_d3,4557_wk1,4557_wk2,4558_d0,4558_d3,4558_wk1,4558_wk2,4559_d0,4559_d3,...,4877_wk2,4936_d0,4936_wk1,4936_wk2,4939_d0,4939_wk1,4939_wk2,5755_d0,5755_d3,5755_wk1
TargetID,,,,,,,,,,,,,,,,,,,,,
BCG_0014c,7.521097,7.413344,7.334556,7.407448,7.393876,7.441137,7.396078,7.464638,7.511167,7.391319,...,7.525532,7.575306,7.434187,7.549169,7.563523,7.498329,7.529780,7.552024,7.524677,7.574339
BCG_0067c,10.113032,10.432085,10.876276,9.853786,10.226141,9.882812,9.549224,9.603722,10.554880,9.554944,...,10.600699,10.600699,10.681497,10.586504,10.839663,10.379874,10.778040,10.312051,10.536067,10.003142
BCG_1596,8.103065,8.223157,8.160017,8.194219,8.291422,8.182801,8.298409,8.456733,8.288878,8.158580,...,8.435111,8.221434,8.397545,8.621949,8.463405,8.623645,8.594069,8.431190,8.294412,8.337461
BCG_3354c,8.382316,8.449695,8.129300,8.168247,8.049271,8.309573,8.118341,8.238133,8.390992,8.379843,...,8.098397,8.239758,8.779252,7.988193,8.233984,8.192326,8.415045,8.665665,8.525736,8.402685
JTY_0020,7.940714,7.850831,7.652419,7.760914,7.714380,7.839520,7.706329,7.955113,7.803979,7.755801,...,8.268402,8.336970,8.063147,8.212019,8.163627,8.289392,7.773844,7.979736,7.841713,8.038801


In [18]:
times = list(set([c.split("_")[1] for c in COUNT_DF]))
times

['wk2', 'd3', 'd0', 'wk1']

In [22]:
# Construct a utility that iterates through each patient and returns TrinaryData
patients = list(set([c.split("_")[0] for c in DF.columns]))

In [33]:
D0 = "d0"
def getDF(patient, full_df=COUNT_DF):
    """
    Constructs TrinaryData for a patient
    """
    columns = [c for c in full_df.columns if patient in c]
    df = full_df[columns].copy()
    df.columns = [c.split("_")[1] for c in df.columns]
    # See if there is a reference time
    if not D0 in df.columns:
        df[D0] = 0
        for column in df.columns:
            df[column] = 0
    else:
        # Construct trinary values
        d0_ser = df[D0].apply(lambda v: np.log2(v))
        df = df.apply(lambda c: np.log2(c) - d0_ser)
        df = df.applymap(lambda v: 1 if v >= 1 else -1 if v <= -1 else 0)
    return df

# Tests
df = getDF("4558")
df

,d0,d3,wk1,wk2
TargetID,,,,
BCG_0014c,0,0,0,0
BCG_0067c,0,0,0,0
BCG_1596,0,0,0,0
BCG_3354c,0,0,0,0
JTY_0020,0,0,0,0
...,...,...,...,...
TBMG_04093,0,0,0,0
TBMG_04094,0,0,0,0
TBMG_04099,0,0,0,0


In [34]:
[p for p in patients if getDF(p).sum().sum() > 0]

[]